# Tutorial 2 — Spectroscopy: AB Pic b

**What you'll learn:** Fitting a medium-resolution K-band spectrum with ForMoSA v2.0.
Compared to Tutorial 1 (photometry), spectroscopy adds *resolution adaptation* —
the model grid must be convolved to match your instrument's spectral resolution —
and allows you to constrain the companion's **radial velocity**.

**Target:** AB Pictoris b — a young (~30 Myr), directly-imaged planetary-mass companion
at ~50 pc orbiting the debris-disk star AB Pic. It is one of the benchmarks for
atmosphere retrieval codes because of its well-characterised orbit and known distance.

**Data:** VLT/SINFONI K-band spectrum (R ≈ 4000, λ = 2.0–2.45 µm).
Published in Palma-Bifani et al. (2023, A&A, 670, A90).

**Grid:** BT-Settl (same grid downloaded in Tutorial 1)

**Estimated runtime:**
- Grid adaptation: < 30 s
- Nested sampling (200 live points, nestle): ~3 min

**References:** Palma-Bifani et al. (2023, A&A, 670, A90)


## Section 0: Setup

In [ ]:
# Cell A: Environment check
import sys
try:
    import ForMoSA
    print(f"ForMoSA {ForMoSA.__version__} — OK")
except ImportError:
    raise ImportError("pip install ForMoSA && conda install dask netCDF4 bottleneck")
print(f"Python {sys.version.split()[0]}")


In [ ]:
# Cell B: Workspace setup
from pathlib import Path
TUTORIAL_DIR = Path(".").resolve()
for d in ["data", "adapted_grid", "results", "grid"]:
    (TUTORIAL_DIR / d).mkdir(exist_ok=True)
print(f"Working directory: {TUTORIAL_DIR}")


In [ ]:
# Cell C: Data download + validation
import urllib.request
from astropy.io import fits

DATA_FILE = TUTORIAL_DIR / "data" / "ABPicb_SINFONI_K.fits"
DATA_URL  = (
    "https://github.com/exoAtmospheres/ForMoSA/releases/download/"
    "tutorial-data-v1/ABPicb_SINFONI_K.fits"
)

if not DATA_FILE.exists():
    print("Downloading observation data...")
    urllib.request.urlretrieve(DATA_URL, DATA_FILE)
    print("Done.")
else:
    print(f"Data already present: {DATA_FILE.name}")

REQUIRED = {"WAV", "WAVE_UNIT", "FLX", "ERR", "RES"}
with fits.open(DATA_FILE) as hdul:
    found = {ext.name for ext in hdul[1:]}

missing = REQUIRED - {k.upper() for k in found}
if missing:
    raise RuntimeError(f"Missing FITS extensions: {missing}")

print("\nFITS extensions (required marked ✓):")
for name in sorted(found):
    mark = "✓" if name.upper() in REQUIRED else " "
    print(f"  {mark} {name}")
print("\nNote: RES contains the spectral resolution R=λ/Δλ per wavelength point.")


In [ ]:
# Cell D: Grid download (skip if already present from Tutorial 1)
import urllib.request
from pathlib import Path

GRID_FILE = TUTORIAL_DIR / "grid" / "BT-Settl.nc"
GRID_URL  = (
    "https://github.com/exoAtmospheres/ForMoSA/releases/download/"
    "tutorial-data-v1/BT-Settl.nc"
)
# Override if you have the grid elsewhere:
# GRID_FILE = Path("/path/to/BT-Settl.nc")

if not GRID_FILE.exists():
    print("Downloading BT-Settl model grid (~1 GB). Keep this file for all tutorials.")
    try:
        from tqdm import tqdm
        class _P(tqdm):
            def update_to(self, b=1, bs=1, ts=None):
                if ts: self.total = ts
                self.update(b * bs - self.n)
        with _P(unit="B", unit_scale=True, desc="BT-Settl.nc") as t:
            urllib.request.urlretrieve(GRID_URL, GRID_FILE, reporthook=t.update_to)
    except ImportError:
        def _p(c, bs, tot):
            print(f"\r  {min(100,c*bs/tot*100):.1f}%", end="", flush=True)
        urllib.request.urlretrieve(GRID_URL, GRID_FILE, reporthook=_p); print()
    print(f"\nSaved: {GRID_FILE}")
else:
    print(f"Grid already present: {GRID_FILE.name}")

import xarray as xr
ds = xr.open_dataset(GRID_FILE, decode_cf=False)
print(f"Grid: {dict(ds.sizes)}")


## Section 1: The science

### Spectroscopy vs photometry

Photometry gives you broad-band colours. A medium-resolution spectrum gives you
the *shape* of molecular absorption features — CO overtone bands at 2.29–2.45 µm,
H₂O bands, and more. This makes spectroscopy far more sensitive to Teff and log g,
and crucially, it allows you to measure the **radial velocity (RV)** of the companion
by cross-correlating the observed features against the model.

### Why no vsini?

To measure rotational broadening (vsini) reliably, you need R > 50,000 so that
individual rotational lines are resolved. SINFONI K-band delivers R ≈ 4000 — enough
for molecular features but not for vsini. We therefore fit `rv` but not `vsini`.
(Tutorial 3 covers vsini with VLT/HiRISE at R ≈ 140,000.)

### AB Pic b

AB Pic b is a ~13 MJup companion at a projected separation of ~275 au, co-moving
with the β Pic moving group (age ≈ 20–30 Myr). Its low surface gravity and warm
temperature make it an L-type object with prominent CO and H₂O absorption.
Literature values: Teff ≈ 1700 K, log g ≈ 4.0, d = 50.1 pc (Hipparcos).


## Section 2: Inspect the data

In [ ]:
from astropy.io import fits
import matplotlib.pyplot as plt
import numpy as np

with fits.open(DATA_FILE) as hdul:
    wav = hdul["WAV"].data.astype(float)   # µm
    flx = hdul["FLX"].data.astype(float)
    err = hdul["ERR"].data.astype(float)
    res = hdul["RES"].data.astype(float)   # spectral resolution R = λ/Δλ

print(f"Wavelength range : {wav.min():.4f} – {wav.max():.4f} µm")
print(f"Number of points : {len(wav)}")
print(f"Resolution R     : {res.mean():.0f} ± {res.std():.0f}  (mean ± std)")

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
ax1.plot(wav, flx, color="#2E86AB", lw=0.8, label="AB Pic b  (SINFONI K)")
ax1.fill_between(wav, flx - err, flx + err, color="#2E86AB", alpha=0.3)
ax1.set_ylabel(r"Flux (W m$^{-2}$ µm$^{-1}$)")
ax1.legend()
ax2.plot(wav, res, color="#E84855", lw=0.8)
ax2.set_xlabel(r"Wavelength (µm)")
ax2.set_ylabel(r"Resolution $R = \lambda / \Delta\lambda$")
plt.tight_layout(); plt.show()


## Section 3: Configure the analysis

In [ ]:
from ForMoSA.config.global_config import ConfigPath, ConfigAdapt, ConfigInversion, ConfigParameters

config_path = ConfigPath(
    observation_path=[str(DATA_FILE)],
    adapt_store_path=str(TUTORIAL_DIR / "adapted_grid"),
    result_path=str(TUTORIAL_DIR / "results"),
    model_path=str(GRID_FILE),
)

# res_cont="500" tells ForMoSA to remove the continuum using a low-pass filter
# at R=500 before comparing model to data. This removes broad-band slope
# differences and lets the fit focus on molecular features.
config_adapt = ConfigAdapt(
    res_cont=["500"],   # enable continuum removal at R=500
)

config_inversion = ConfigInversion(
    wav_fit=["2.0, 2.45"],   # K-band window (µm) — avoids noisy edges
    ns_algo="nestle",
    npoints=200,             # more live points for better posterior sampling
    logL_type=["chi2"],
)

config_params = ConfigParameters(
    par1=["uniform", "1200", "3000"],  # Teff (K)
    par2=["uniform", "2.5", "5.5"],   # log g (dex)
    r=["uniform", "0.5", "3.0"],      # radius (R_Jupiter)
    d=["constant", "50.1"],           # distance (pc) — Hipparcos, fixed
    rv=["uniform", "-100", "100"],    # radial velocity (km/s)
    # vsini not fitted — SINFONI K (R≈4000) cannot resolve rotational broadening
)

print("Configuration:")
print(f"  wav_fit : {config_inversion.wav_fit[0]} µm")
print(f"  Free    : par1 (Teff), par2 (log g), r, rv")
print(f"  Fixed   : d = 50.1 pc")


## Section 4: Adapt the grid

In [ ]:
from ForMoSA import Analysis

adapted = False  # set True to skip on re-runs

analysis = Analysis(config_path, adapted=adapted, fitted=False)

if not adapted:
    # Adaptation convolves each model spectrum to match the SINFONI resolution
    # and resamples it onto the observed wavelength grid.
    print("Adapting grid (convolution + resampling)...")
    analysis.adapt(config_adapt, config_inversion)
    print("Done. Set adapted=True to skip on re-runs.")


## Section 5: Run the nested sampling fit

In [ ]:
from ForMoSA.config.global_config import Config_NS

config_ns = Config_NS()

print(f"Running {config_inversion.ns_algo} with {config_inversion.npoints} live points...")
analysis.nested_sampling(config_params, config_adapt, config_inversion, config_NS=config_ns)
print("\nFit complete.")


## Section 6: Results

In [ ]:
analysis.plot(analysis.ns.results, plot_native_model=False)
print(analysis.ns.results.summary(sigma=1))


## Section 7: INI file alternative

Same workflow as Tutorial 1 — generate a template `.ini`, edit it, and load with `ConfigLoader`.


In [ ]:
from ForMoSA.config.global_config import ConfigGenerator

generator = ConfigGenerator()
generator.save(str(TUTORIAL_DIR), "config.ini")
print(f"Template written to: {TUTORIAL_DIR / 'config.ini'}")
print("Edit the file, then load with ConfigLoader (see Tutorial 1, Section 7 for the full pattern).")


## Section 8: Next steps

- **Tutorial 3 — HCHR mode (AF Lep b):** VLT/HiRISE at R ≈ 140,000.
  Introduces `STAR_FLUX` extension, high-contrast modeling, and vsini.
- **Tutorial 5 — Advanced plotting:** Deep-dive into every ForMoSA plot
  and how to customise it for publication. Uses these results as input.
